In [2]:
import numpy as np 
import pandas as pd 

In [3]:
import seaborn as sns 
import matplotlib.pyplot as plt 

In [4]:
df_train = pd.read_csv("customer_train_dataset.csv")

In [5]:
df_test = pd.read_csv("customer_prediction_dataset.csv")

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
print(df_train.info())

In [ ]:
df_train.isnull().sum()

In [ ]:
df_test.isnull().sum()

In [ ]:
sns.heatmap(df_train.corr(numeric_only=True), cmap="viridis")

In [ ]:
df_train.columns

In [ ]:
columns = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available",
          "Range_Anxiety_Level", "Will_Buy_EV"]
for column in columns:
    print(f"Unique Values of {column}")
    print(df_train[column].unique())
    print(f"----------------")

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df_train.drop("Will_Buy_EV", axis=1)
y = df_train["Will_Buy_EV"]
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.25, random_state=42)

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
columns = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available",
          "Range_Anxiety_Level"]

In [ ]:
encoders = {}

for col in columns:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    encoders[col] = le

In [ ]:
label = LabelEncoder()
y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [ ]:
X_train_scaled = X_train_scaled.drop("id", axis=1)
X_test_scaled = X_test_scaled.drop("id", axis=1)

In [ ]:
X_test_scaled

In [ ]:
X_train_scaled

In [ ]:
figure = plt.figure(figsize=(16,12))
sns.heatmap(X_train_scaled.corr(), annot=True, cmap="viridis")
plt.show()

In [ ]:
from xgboost import XGBClassifier 

In [ ]:
xgb = XGBClassifier() 
xgb.fit(X_train_scaled, y_train)

In [ ]:
y_pred = xgb.predict(X_test_scaled)

In [ ]:
print("confussion matrix \n: ", confusion_matrix(y_test, y_pred))
print("classification report \n: ", classification_report(y_test, y_pred))
print("accuracy score \n: ", accuracy_score(y_test, y_pred))

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)
y_pred = lgbm.predict(X_test)
print("confussion matrix \n: ", confusion_matrix(y_test, y_pred))
print("classification report \n: ", classification_report(y_test, y_pred))
print("accuracy score \n: ", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rfc = RandomForestClassifier()

In [ ]:
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)
print("confussion matrix \n: ", confusion_matrix(y_test, y_pred))
print("classification report \n: ", classification_report(y_test, y_pred))
print("accuracy score \n: ", accuracy_score(y_test, y_pred))

Hyperparameter tuning are done by my computer. There is no significant change in accuracy score.

In [ ]:
columns = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available",
          "Range_Anxiety_Level"]
for column in columns:
    print(f"Unique Values of {column}")
    print(df_test[column].unique())
    print(f"----------------")

In [ ]:
df_test

In [ ]:
test_ids = df_test["id"]

In [ ]:
df_test_wo_id = df_test.copy()

In [ ]:
df_test_wo_id.columns

In [ ]:
columns = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available",
          "Range_Anxiety_Level"]
for col in columns:
    df_test_wo_id[col] = encoders[col].transform(df_test_wo_id[col])

In [ ]:
df_test_wo_id

In [ ]:
df_test_scaled = scaler.transform(df_test_wo_id)

In [ ]:
df_test_scaled = pd.DataFrame(df_test_scaled, columns=df_test.columns)

In [ ]:
df_test_scaled.drop("id", axis=1,inplace=True)

In [ ]:
probabilities = xgb.predict_proba(df_test_scaled)

In [ ]:
probabilities

In [ ]:
test_ids

In [ ]:
result_df = pd.DataFrame({
    'id': test_ids,
    'prob_0_no': probabilities[:, 0],
    'prob_1_yes': probabilities[:, 1]
})

In [ ]:
result_df